# T03 — Feature Engineering
**Input:** `data/loaded/train_rul.csv`, `data/loaded/test_rul.csv`  
**Goal:** drop constant sensors → normalize per (dataset_id, op_cluster) → add rolling features  
**Output:** `data/processed/train_features.csv`, `data/processed/test_features.csv`  
**Artifacts:** `artifacts/kmeans_op_clusters.pkl`, `artifacts/scalers.pkl`  
**Next:** modeling notebooks (ARIMA, LSTM, TFT, ...)

In [9]:
import sys
import os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

Project root: /Users/dhruvparmar/DAU/sem_2/IT_402_Applied_Forecasting_Methods/Project/Aircraft Engine Failure Forecasting


In [10]:
import pandas as pd
from src.preprocessing.cleaning import drop_sensors, get_sensor_cols
from src.preprocessing.scaling import (
    scale_sensors,
    verify_scaling,
    save_scaling_artifacts,
)
from src.features.rolling import add_rolling_features, verify_rolling_features

LOADED_DIR      = ROOT / "data" / "loaded"
PROC_DIR     = ROOT / "data" / "processed"
ARTIFACTS_DIR = ROOT / "artifacts"
PROC_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

## Load from T02 outputs

In [11]:
train = pd.read_csv(LOADED_DIR / "train_rul.csv")
test  = pd.read_csv(LOADED_DIR / "test_rul.csv")
print(f"train: {train.shape}  |  test: {test.shape}")

train: (61249, 27)  |  test: (41214, 27)


---
## Step 1 — Drop low-variance sensors
detected on train only (never test) — per-dataset_id variance check  
a sensor must be flat in ALL subsets to be dropped (not just FD001)

In [12]:
train, test, dropped_sensors = drop_sensors(train, test)
sensor_cols = get_sensor_cols(train)
print(f"\nSensors kept ({len(sensor_cols)}): {sensor_cols}")

  dropped 1 sensors: ['s16']
  kept    20 sensors: ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's17', 's18', 's19', 's20', 's21']

Sensors kept (20): ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's17', 's18', 's19', 's20', 's21']


---
## Step 2 — Normalize per (dataset_id, op_cluster) using StandardScaler
StandardScaler chosen over MinMaxScaler: test values can exceed train extremes  
as engines degrade — StandardScaler handles this as high z-scores, not out-of-range values  
KMeans n_clusters adapts automatically 
fitted artifacts saved to `artifacts/` for inference reuse

In [13]:
train, test, km, scalers, sensor_cols = scale_sensors(train, test, sensor_cols)
verify_scaling(train, sensor_cols)

# persist so downstream inference / other notebooks don't need to refit
save_scaling_artifacts(km, scalers, ARTIFACTS_DIR)

  [INFO] Dropping sensors constant in at least one cluster: ['s5', 's19', 's1', 's18']
  fitted 6 StandardScalers across 6 op_clusters
  [PASS] per-cluster means < 0.05 and stds > 0 for all 6 op_clusters
  [INFO] global sensor std range: [1.0000, 1.0000]
  saved kmeans_op_clusters.pkl and scalers.pkl → /Users/dhruvparmar/DAU/sem_2/IT_402_Applied_Forecasting_Methods/Project/Aircraft Engine Failure Forecasting/artifacts


/opt/anaconda3/envs/dl/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/dl/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/dl/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/dl/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/opt/anaconda3/envs/dl/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/opt/anaconda3/envs/dl/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: invalid value encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/dhruv

In [14]:
# op_cluster distribution — FD001/FD003 should use fewer clusters than FD002/FD004
assert "op_cluster" in train.columns and "op_cluster" in test.columns
print("op_cluster distribution (train):")
print(train["op_cluster"].value_counts().sort_index().to_string())

op_cluster distribution (train):
op_cluster
0    15395
1     9224
2     9139
3     9091
4     9238
5     9162


---
## Step 3 — Rolling features
input is sorted by (engine_id, cycle) inside add_rolling_features  
rolling windows never cross engine boundaries

In [15]:
ROLLING_WINDOWS = [5, 10, 20]  # 20 added: covers full LSTM window at every position

train = add_rolling_features(train, sensor_cols, windows=ROLLING_WINDOWS)
test  = add_rolling_features(test,  sensor_cols, windows=ROLLING_WINDOWS)

n_rolling = len(sensor_cols) * len(ROLLING_WINDOWS) * 2
print(f"Added {n_rolling} rolling feature columns")
print(f"train: {train.shape}  |  test: {test.shape}")

Added 96 rolling feature columns
train: (61249, 123)  |  test: (41214, 123)


## Verification

In [16]:
# no NaN anywhere
assert train.isnull().sum().sum() == 0, "NaN in train"
assert test.isnull().sum().sum()  == 0, "NaN in test"
print("[PASS] no NaN values")

# dropped sensors are gone
for s in dropped_sensors:
    assert s not in train.columns, f"{s} still in train"
print(f"[PASS] all {len(dropped_sensors)} dropped sensors absent")

# rul_last must not be present (dropped in T02, verified again here for safety)
assert "rul_last" not in train.columns and "rul_last" not in test.columns
print("[PASS] rul_last absent — no leakage risk")

# RUL values must be unchanged from T02 input
train_rul_check = pd.read_csv(LOADED_DIR / "train_rul.csv")["RUL"]
# T03 re-sorts by (engine_id, cycle) internally — align on engine_id+cycle not position
ref = pd.read_csv(LOADED_DIR / "train_rul.csv")[["engine_id", "cycle", "RUL"]]
check = train[["engine_id", "cycle", "RUL"]]
merged = check.merge(ref, on=["engine_id", "cycle"], suffixes=("_new", "_ref"))
assert (merged["RUL_new"] == merged["RUL_ref"]).all(), "RUL values changed during feature engineering!"
print("[PASS] RUL values unchanged")

# rolling feature integrity
verify_rolling_features(train, sensor_cols, ROLLING_WINDOWS)

[PASS] no NaN values
[PASS] all 1 dropped sensors absent
[PASS] rul_last absent — no leakage risk
[PASS] RUL values unchanged
  [PASS] all 96 rolling feature columns present
  [PASS] no NaN values in rolling features
  [PASS] rolling std is 0 at first cycle of each engine


## Save

In [17]:
train.to_csv(PROC_DIR / "train_features.csv", index=False)
test.to_csv(PROC_DIR / "test_features.csv", index=False)
print(f"Saved: {PROC_DIR / 'train_features.csv'}")
print(f"Saved: {PROC_DIR / 'test_features.csv'}")

Saved: /Users/dhruvparmar/DAU/sem_2/IT_402_Applied_Forecasting_Methods/Project/Aircraft Engine Failure Forecasting/data/processed/train_features.csv
Saved: /Users/dhruvparmar/DAU/sem_2/IT_402_Applied_Forecasting_Methods/Project/Aircraft Engine Failure Forecasting/data/processed/test_features.csv


## Feature summary for modeling notebooks

In [18]:
rolling_cols = [c for c in train.columns if "_rmean_" in c or "_rstd_" in c]
all_feature_cols = sensor_cols + rolling_cols

print(f"Raw sensor features:    {len(sensor_cols)}")
print(f"Rolling features:       {len(rolling_cols)}")
print(f"Total feature columns:  {len(all_feature_cols)}")
print(f"\nNon-feature columns kept: engine_id, cycle, op1, op2, op3, dataset_id, op_cluster, RUL")
print(f"\nAll feature cols for X: {all_feature_cols}")

Raw sensor features:    16
Rolling features:       96
Total feature columns:  112

Non-feature columns kept: engine_id, cycle, op1, op2, op3, dataset_id, op_cluster, RUL

All feature cols for X: ['s2', 's3', 's4', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21', 's2_rmean_5', 's2_rstd_5', 's3_rmean_5', 's3_rstd_5', 's4_rmean_5', 's4_rstd_5', 's6_rmean_5', 's6_rstd_5', 's7_rmean_5', 's7_rstd_5', 's8_rmean_5', 's8_rstd_5', 's9_rmean_5', 's9_rstd_5', 's10_rmean_5', 's10_rstd_5', 's11_rmean_5', 's11_rstd_5', 's12_rmean_5', 's12_rstd_5', 's13_rmean_5', 's13_rstd_5', 's14_rmean_5', 's14_rstd_5', 's15_rmean_5', 's15_rstd_5', 's17_rmean_5', 's17_rstd_5', 's20_rmean_5', 's20_rstd_5', 's21_rmean_5', 's21_rstd_5', 's2_rmean_10', 's2_rstd_10', 's3_rmean_10', 's3_rstd_10', 's4_rmean_10', 's4_rstd_10', 's6_rmean_10', 's6_rstd_10', 's7_rmean_10', 's7_rstd_10', 's8_rmean_10', 's8_rstd_10', 's9_rmean_10', 's9_rstd_10', 's10_rmean_10', 's10_rstd_10', 's11_rmean_10',